# 01 — Inventory and provenance

Enumerate what is on disk under `data/raw/` (and the Mobility catalog cache), tie paths to `configs/`, and read the latest download manifest under `artifacts/logs/provenance/`. Use this notebook after `python scripts/download_data.py` so provenance matches the files you see.

**Run from repo root** (or any subfolder—`find_repo_root()` walks up to locate `configs/san_diego.yaml`).

**Next:** [`02_gtfs_schedule_exploration.ipynb`](02_gtfs_schedule_exploration.ipynb)

**Docs:** Update `context/DATASETS.md` (Last downloaded / Feed version) using the tables below.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "configs" / "san_diego.yaml").exists():
            return d
    raise FileNotFoundError(
        "Could not find configs/san_diego.yaml. cd to the repo root or start Jupyter there."
    )


def human_size(b: int) -> str:
    if b < 1024:
        return f"{b} B"
    if b < 1024**2:
        return f"{b / 1024:.1f} KB"
    if b < 1024**3:
        return f"{b / 1024**2:.1f} MB"
    return f"{b / 1024**3:.1f} GB"


REPO_ROOT = find_repo_root()
CITY_YAML = REPO_ROOT / "configs" / "san_diego.yaml"
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = C:\Users\sardo\OneDrive\Desktop\Classes\Projects\BayesTransitEquity


## Merged config (`defaults.yaml` + city YAML)

Same merge rule as `scripts/download_data.py`.

In [2]:
with open(REPO_ROOT / "configs" / "defaults.yaml") as f:
    config = yaml.safe_load(f)
with open(CITY_YAML) as f:
    config.update(yaml.safe_load(f))

city = config.get("city_label", config.get("city"))
bbox = config.get("bbox")
state_fips = config.get("state_fips")
county_fips = config.get("county_fips")
census_cfg = config.get("census", {})
acs_year = census_cfg.get("acs_year", 2023)

summary = pd.DataFrame(
    [
        {"key": "city", "value": city},
        {"key": "bbox [min_lon, min_lat, max_lon, max_lat]", "value": str(bbox)},
        {"key": "state_fips", "value": state_fips},
        {"key": "county_fips", "value": county_fips},
        {"key": "acs_year", "value": acs_year},
    ]
)
display(summary)

agencies = pd.DataFrame(config.get("gtfs_agencies", []))
display(Markdown("### GTFS agencies (config)"))
display(agencies[[c for c in ["id", "name", "mobility_db_id", "raw_path", "download_url"] if c in agencies.columns]])

mc = config.get("mobility_catalog") or {}
display(Markdown("### mobility_catalog (defaults)"))
display(pd.Series(mc, name="value"))

,key,value
0,city,"San Diego, CA"
1,"bbox [min_lon, min_lat, max_lon, max_lat]","[-117.28, 32.53, -116.93, 33.11]"
2,state_fips,06
3,county_fips,073
4,acs_year,2023


### GTFS agencies (config)

,id,name,mobility_db_id,raw_path,download_url
0,mts,San Diego Metropolitan Transit System,mdb-13,data/raw/gtfs/mts/,http://www.sdmts.com/google_transit_files/goog...
1,nctd,North County Transit District,mdb-14,data/raw/gtfs/nctd/,http://www.gonctd.com/google_transit.zip


### mobility_catalog (defaults)

feeds_csv_url          https://files.mobilitydatabase.org/feeds_v2.csv
cache_relative_path        data/interim/mobility_database/feeds_v2.csv
max_age_days                                                         7
Name: value, dtype: object

## Latest provenance manifest

Written by `scripts/download_data.py` to `artifacts/logs/provenance/data_manifest_<city>_<timestamp>.json`.

In [3]:
prov_dir = REPO_ROOT / "artifacts" / "logs" / "provenance"
manifest_paths = sorted(
    prov_dir.glob("data_manifest_*.json"), key=lambda p: p.stat().st_mtime
)
if not manifest_paths:
    display(Markdown("_No manifest found. Run `python scripts/download_data.py` first._"))
    manifest = None
else:
    latest_m = manifest_paths[-1]
    display(Markdown(f"**Latest manifest:** `{latest_m.relative_to(REPO_ROOT)}`"))
    with open(latest_m, encoding="utf-8") as f:
        manifest = json.load(f)
    display(
        pd.DataFrame(
            [
                {"field": "generated_at", "value": manifest.get("generated_at")},
                {"field": "city", "value": manifest.get("city")},
                {"field": "data_freeze_date", "value": manifest.get("data_freeze_date")},
            ]
        )
    )

**Latest manifest:** `artifacts\logs\provenance\data_manifest_san_diego_2026-03-29_2055.json`

,field,value
0,generated_at,2026-03-29T20:55:11.761101+00:00
1,city,san_diego
2,data_freeze_date,2026-03-29


In [4]:
if manifest:
    rows = []
    for s in manifest.get("sources", []):
        rows.append(
            {
                "label": s.get("label"),
                "source_type": s.get("source_type"),
                "status": s.get("status"),
                "dest": s.get("dest"),
                "url": (s.get("url") or "")[:80] + ("..." if len(s.get("url") or "") > 80 else ""),
                "size_bytes": s.get("size_bytes"),
                "md5": (s.get("md5") or "")[:12] + "..." if s.get("md5") else None,
                "downloaded_at": s.get("downloaded_at"),
            }
        )
    df_src = pd.DataFrame(rows)
    if "size_bytes" in df_src.columns:
        df_src["size"] = df_src["size_bytes"].apply(
            lambda x: human_size(int(x)) if pd.notna(x) and x is not None else ""
        )
    display(Markdown("### All sources in manifest"))
    display(df_src.drop(columns=["size_bytes"], errors="ignore"))

### All sources in manifest

,label,source_type,status,dest,url,md5,downloaded_at,size
0,GTFS/mts,gtfs_schedule,downloaded,data\raw\gtfs\mts\google_transit.zip,https://files.mobilitydatabase.org/mdb-13/late...,8c69875a2a9e...,2026-03-29T20:55:11.119848+00:00,4.6 MB
1,GTFS/nctd,gtfs_schedule,downloaded,data\raw\gtfs\nctd\google_transit.zip,https://files.mobilitydatabase.org/mdb-14/late...,089bd6d961a0...,2026-03-29T20:55:11.660858+00:00,1.6 MB


In [5]:
if manifest:
    gtfs_rows = [s for s in manifest.get("sources", []) if s.get("source_type") == "gtfs_schedule"]
    if gtfs_rows:
        cols = [
            "agency_id",
            "agency_name",
            "mobility_db_id",
            "gtfs_download_url_source",
            "feed_version",
            "feed_start_date",
            "feed_end_date",
            "status",
            "md5",
        ]
        g = pd.DataFrame([{c: r.get(c) for c in cols} for r in gtfs_rows])
        g["n_extracted"] = [len(r.get("extracted_files") or []) for r in gtfs_rows]
        display(Markdown("### GTFS schedule (from manifest + feed_info)"))
        display(g)

### GTFS schedule (from manifest + feed_info)

,agency_id,agency_name,mobility_db_id,gtfs_download_url_source,feed_version,feed_start_date,feed_end_date,status,md5,n_extracted
0,mts,San Diego Metropolitan Transit System,mdb-13,mobility_catalog.urls.latest,2601 unmerged v2,20260125,20260606,downloaded,8c69875a2a9e0c985bec49066f07b559,20
1,nctd,North County Transit District,mdb-14,mobility_catalog.urls.latest,Version_20250205,,,downloaded,089bd6d961a08b7b307fe07a7ffadde6,17


## On-disk file inventory

Recursive listing of `data/raw/` and `data/interim/mobility_database/` (catalog cache). MD5 on full tree can be slow; we show size only here.

In [6]:
inv_roots = [
    REPO_ROOT / "data" / "raw",
    REPO_ROOT / "data" / "interim" / "mobility_database",
]
file_rows = []
for base in inv_roots:
    if not base.exists():
        continue
    for p in base.rglob("*"):
        if p.is_file() and p.name != ".gitkeep":
            rel = p.relative_to(REPO_ROOT)
            st = p.stat()
            file_rows.append(
                {
                    "path": str(rel).replace("\\", "/"),
                    "bytes": st.st_size,
                    "modified_utc": pd.Timestamp(st.st_mtime, unit="s", tz="UTC").isoformat(),
                }
            )

df_inv = pd.DataFrame(file_rows).sort_values("path").reset_index(drop=True)
df_inv["size"] = df_inv["bytes"].map(human_size)
display(Markdown(f"### {len(df_inv)} files"))
display(df_inv[["path", "size", "modified_utc"]])

### 49 files

,path,size,modified_utc
0,data/interim/mobility_database/feeds_v2.csv,2.3 MB,2026-03-29T20:55:09.900208950+00:00
1,data/raw/census/acs5_2023_sd_county.json,135.9 KB,2026-03-29T19:40:48.985675097+00:00
2,data/raw/census/tl_2023_06_tract.zip,31.0 MB,2026-03-29T19:40:46.791359186+00:00
3,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,5 B,2026-03-29T19:40:46.908233166+00:00
4,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,1.1 MB,2026-03-29T19:40:46.917984009+00:00
5,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,165 B,2026-03-29T19:40:46.917984009+00:00
6,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,49.2 MB,2026-03-29T19:40:47.362096071+00:00
7,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,22.6 KB,2026-03-29T19:40:47.362096071+00:00
8,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,52.3 KB,2026-03-29T19:40:47.362096071+00:00
9,data/raw/census/tl_2023_06_tract/tl_2023_06_tr...,71.4 KB,2026-03-29T19:40:47.362096071+00:00


## Census artifacts (paths derived from config + download script conventions)

- TIGER: state tract file `tl_{acs_year}_{state_fips}_tract.zip` (filter to county `{county_fips}` in later notebooks).
- ACS: `acs5_{acs_year}_sd_county.json` (tract rows for San Diego County).

In [7]:
census_dir = REPO_ROOT / "data" / "raw" / "census"
tiger_name = f"tl_{acs_year}_{state_fips}_tract.zip"
tiger_zip = census_dir / tiger_name
tiger_shp_dir = census_dir / tiger_name.replace(".zip", "")
acs_path = census_dir / f"acs5_{acs_year}_sd_county.json"

census_check = pd.DataFrame(
    [
        {"artifact": "TIGER zip", "path": str(tiger_zip.relative_to(REPO_ROOT)), "exists": tiger_zip.exists()},
        {
            "artifact": "TIGER extracted dir",
            "path": str(tiger_shp_dir.relative_to(REPO_ROOT)),
            "exists": tiger_shp_dir.is_dir(),
        },
        {"artifact": "ACS JSON", "path": str(acs_path.relative_to(REPO_ROOT)), "exists": acs_path.exists()},
    ]
)
display(census_check)

if acs_path.exists():
    acs_data = json.loads(acs_path.read_text(encoding="utf-8"))
    n_tracts = max(0, len(acs_data) - 1) if isinstance(acs_data, list) else 0
    display(Markdown(f"**ACS tract rows (excl. header):** {n_tracts}"))
    if acs_data and isinstance(acs_data[0], list):
        display(Markdown(f"**ACS columns (header):** `{acs_data[0][:8]}...`"))

,artifact,path,exists
0,TIGER zip,data\raw\census\tl_2023_06_tract.zip,True
1,TIGER extracted dir,data\raw\census\tl_2023_06_tract,True
2,ACS JSON,data\raw\census\acs5_2023_sd_county.json,True


**ACS tract rows (excl. header):** 737

**ACS columns (header):** `['NAME', 'B01003_001E', 'B01003_001M', 'B08201_002E', 'B08201_002M', 'B19013_001E', 'B19013_001M', 'B17001_002E']...`

## Expected vs missing (for later EDA notebooks)

| Notebook | Typical raw inputs |
|----------|-------------------|
| 05 OSM | `data/raw/osm/sd_walk_network.graphml` from `--sources osm` |
| 06 jobs | `data/raw/external/lodes/*.csv.gz` from `--sources lodes` |
| 06 POIs | Often queried live via OSMnx/Overpass in the notebook |

Below: quick existence check for those paths.

In [8]:
osm_gml = REPO_ROOT / "data" / "raw" / "osm" / "sd_walk_network.graphml"
lodes_dir = REPO_ROOT / "data" / "raw" / "external" / "lodes"
lodes_files = list(lodes_dir.glob("*")) if lodes_dir.is_dir() else []

gaps = pd.DataFrame(
    [
        {"item": "OSM walk graph (05)", "path": str(osm_gml.relative_to(REPO_ROOT)), "ok": osm_gml.is_file()},
        {
            "item": "LODES WAC (06 jobs)",
            "path": str(lodes_dir.relative_to(REPO_ROOT)),
            "ok": bool(lodes_files),
        },
    ]
)
display(gaps)
if not osm_gml.is_file() or not lodes_files:
    display(
        Markdown(
            "Run: `python scripts/download_data.py --config configs/san_diego.yaml --sources osm lodes`"
        )
    )
if lodes_files:
    display(pd.DataFrame([{"path": f.relative_to(REPO_ROOT), "size": human_size(f.stat().st_size)} for f in lodes_files]))

,item,path,ok
0,OSM walk graph (05),data\raw\osm\sd_walk_network.graphml,False
1,LODES WAC (06 jobs),data\raw\external\lodes,False


Run: `python scripts/download_data.py --config configs/san_diego.yaml --sources osm lodes`

## Export artifacts

Tables exported to `artifacts/tables/` following the naming convention:
`eda__<description>__<YYYY-MM-DD>.csv`

In [9]:
from datetime import date as _date

TODAY = _date.today().isoformat()
ART_TABLES = REPO_ROOT / "artifacts" / "tables"
ART_TABLES.mkdir(parents=True, exist_ok=True)

exported = []

# ── File inventory ────────────────────────────────────────────────────────────
if "df_inv" in dir() and df_inv is not None and not df_inv.empty:
    p = ART_TABLES / f"eda__file_inventory__{TODAY}.csv"
    df_inv.to_csv(p, index=False)
    exported.append(str(p.relative_to(REPO_ROOT)))

# ── GTFS feed info (from manifest) ───────────────────────────────────────────
if manifest:
    gtfs_rows = [s for s in manifest.get("sources", []) if s.get("source_type") == "gtfs_schedule"]
    if gtfs_rows:
        cols = [
            "agency_id", "agency_name", "mobility_db_id",
            "gtfs_download_url_source", "feed_version",
            "feed_start_date", "feed_end_date", "status", "md5", "size_bytes",
        ]
        gdf = pd.DataFrame([{c: r.get(c) for c in cols} for r in gtfs_rows])
        gdf["n_extracted_files"] = [len(r.get("extracted_files") or []) for r in gtfs_rows]
        p = ART_TABLES / f"eda__gtfs_feed_info__{TODAY}.csv"
        gdf.to_csv(p, index=False)
        exported.append(str(p.relative_to(REPO_ROOT)))

    # ── Manifest summary ──────────────────────────────────────────────────────
    msummary = pd.DataFrame([
        {"field": "generated_at",    "value": manifest.get("generated_at")},
        {"field": "city",            "value": manifest.get("city")},
        {"field": "data_freeze_date","value": manifest.get("data_freeze_date")},
        {"field": "n_sources",       "value": len(manifest.get("sources", []))},
    ])
    p = ART_TABLES / f"eda__manifest_summary__{TODAY}.csv"
    msummary.to_csv(p, index=False)
    exported.append(str(p.relative_to(REPO_ROOT)))

# ── Census artifact check table ───────────────────────────────────────────────
if "census_check" in dir() and census_check is not None:
    p = ART_TABLES / f"eda__census_artifact_check__{TODAY}.csv"
    census_check.to_csv(p, index=False)
    exported.append(str(p.relative_to(REPO_ROOT)))

display(Markdown("### Exported"))
for e in exported:
    display(Markdown(f"- `{e}`"))
print(f"\n{len(exported)} artifact(s) written to artifacts/tables/")

### Exported

- `artifacts\tables\eda__file_inventory__2026-03-29.csv`

- `artifacts\tables\eda__gtfs_feed_info__2026-03-29.csv`

- `artifacts\tables\eda__manifest_summary__2026-03-29.csv`

- `artifacts\tables\eda__census_artifact_check__2026-03-29.csv`


4 artifact(s) written to artifacts/tables/
